In [ ]:
# GAUSS ELIMINATION METHOD IN PYTHON
import numpy as np

def gaussElim(A, b):
    #contingencies (error handling)
    if A.shape[0] != A.shape[1]:
        print("Square matrix not given\n")
        return
    
    if b.shape[1] > 1 or b.shape[0] != b.shape[0]:
        print("Constant vector incorrectly sized\n")
        return
    
    #initialization
    n = len(b)
    m = n - 1
    i = 0
    j = i - 1
    x = np.zeros(n)
    newLine = '\n'
    
    #create augmented matrix
    augmentedMatrix = np.concatenate((A, b), axis=1, dtype=float)
    print(f"Augmented matrix: {newLine}{augmentedMatrix}")
    
    #apply gauss elimination
    while i < n:
        if augmentedMatrix[i][i] == 0.0:
            print("Divide by zero error")
            return
        
        for j in range(i + 1, n):
            scalingFactor = augmentedMatrix[j][i]/augmentedMatrix[i][i]
            augmentedMatrix[j] = augmentedMatrix[j] - (scalingFactor * augmentedMatrix[i])
            print(augmentedMatrix) #for displaying the process
            
        i = i + 1
        
    #backwards substitution
    x[m] = augmentedMatrix[m][m]
    
    for k in range(n-2, -1, -1):
        x[k] = augmentedMatrix[k][n]
        
        for j in range(k+1, n):
            x[k] = x[k] - augmentedMatrix[k][j] * x[j]
        x[k] = x[k] / augmentedMatrix[k][k]
        
    #output result
    for answer in range(n):
        print(f"x{answer} = {x[answer]}")
        
# PROBLEM EXAMPLE        
A = np.array([[
    [0, 0, 2, 1, 2],
    [0, 1, 0, 2, -1],
    [1, 2, 0, -2, 0],
    [0, 0, 0, -1, 1],
    [0, 1, -1, 1, -1]
    ]], dtype=float)

b = np.array([1, 1, -4, -2, -1], dtype=float)

gaussElim(A, b)

In [ ]:
# GAUSS-JORDAN METHOD IN PYTHON
def gaussJordan(A, b):
    n = len(A)
    M = A

    i = 0
    for x in M:
        x.append(b[i])
        i += 1

    for k in range(n):
        for i in range(k, n):
            if abs(M[i][k]) > abs(M[k][k]):
                M[k], M[i] = M[i], M[k]
            else:
                pass

        for j in range(k+1, n):
            q = M[j][k] / M[k][k]
            for m in range(k, n+1):
                M[j][m] -= q * M[k][m]

    x = [0 for i in range(n)]

    x[n-1] = M[n-1][n] / M[n-1][n-1]
    for i in range(n-1, -1, -1):
        z = 0
        for j in range(i+1, n):
            z = z  + M[i][j]*x[j]
        x[i] = (M[i][n] - z) / M[i][i]
    return x

# PROBLEM EXAMPLE
A = np.array([[
    [2, -3, 1],
    [-3, 2, -5],
    [2, 4, -1]
    ]], dtype=float)

b = np.array([3, -9, -5], dtype=float)

gaussJordan(A, b)

In [ ]:
# LU DECOMPOSITION IN PYTHON
import numpy as np

def LUDecompo(A):
    n = A.shape[0]
    L = np.zeros((n, n))
    U = np.zeros((n, n))
    
    for i in range(n):
        # upper triangular
        for k in range(i, n):
            sum = sum(L[i][j] * U[j][k] for j in range(i))
            U[i][k] = A[i][k] - sum
            
        # lower triangular
        L[i][i] = 1 # diagonal as 1
        for k in range(i + 1, n):
            sum = sum(L[k][j] * U[j][i] for j in range(i))
            if U[i][i] == 0:
                raise ZeroDivisionError("Zero pivot encountered.")
            L[k][i] = (A[k][i] - sum) / U[i][i]
        
    return L, U
        
# PROBLEM EXAMPLE
A = np.array([
            [4, -1, 0],
            [-1, 4, -1],
            [0, -1, 4]
            ], dtype=float)

L, U = LUDecompo(A)

print(f"Lower Triangular matrix: {L}\n")
print(f"Upper Triangular matrix: {U}\n")

In [ ]:
# GAUSS-SEIDEL METHOD IN PYTHON
import numpy as np

def isDiagonallyDominant(x): #to check if its diagonally dominant
    x = np.array(x) #change it into an array
    if x.ndim != 2 or x.shape[0] != x.shape[1]: #x.ndim != 2 ensures that the matrix is 2D, x.shape[0] != x.shape[1] ensures that it's a square matrix. DIAGONALLY DOMINANCE IS ONLY DEFINED AT SQUARE MATRICES.
        return False
    x = np.abs(x) #change every value in the matrix into absolute
    diag = np.diag(x) #it just takes the diagonal
    off_diag = np.sum(x, axis=1) - diag #axis=1 to take one row, sum all in one row and subtract it with the diagonal
    return np.all(diag > off_diag) #check if the diagonal is larger than the off-diagonal

def gaussSeidel(A, b, threshold=0.01, t=15):
    print(f"A: {A}, y: {b}")
    if not isDiagonallyDominant(A):
        print("Matrix is not diagonally dominant.\n")
        return
    
    #change the list into an array
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float)
    
    diag = np.diag(A) #take the diagonal for division
    A = -A #negate A
    np.fill_diagonal(A, 0) #fill diagonal A with 0 so the multiplication works (the result gotta be 0)
    
    x_old = np.zeros(A.shape[0]) #[x1, x2, x3] = [0, 0, 0]
    
    #iteration based on max iteration (t)
    for i in range(t):
        x_new = np.array(x_old) #copy the old x to the new x
        
        for idx, row in enumerate(A):
            x_new[idx] = (b[idx] + np.dot(row, x_new)) / diag[idx]
            
        print(f"Iteration: {i + 1}", x_new)
        
        #threshold check
        dx = np.sqrt(np.dot(x_new - x_old, x_new - x_old))
        if dx < threshold:
            print("Convergent.\n")
            return
        
        x_old = x_new
        
    print("Not convergent.\n")
    
# PROBLEM EXAMPLE(S)
x1 = np.array()
y1 = np.array()
gaussSeidel(x1, y1)

x2 = np.array()
y2 = np.array()
gaussSeidel(x2, y2)